# YAML + Colab 학습
새 실험은 이 노트북과 YAML만 추가합니다. 공통 코드는 GitHub에서 가져옵니다.
Colab에서 GPU를 선택하고 1번부터 실행하세요. 처음에는 smoke.yaml로 확인합니다.
W&B 키는 Colab Secrets의 WANDB_API_KEY에, 비공개 GitHub 토큰은 GITHUB_TOKEN에 등록하세요.


## 1번 셀 · GitHub에서 코드 가져오기
REPOSITORY를 내 저장소로 바꾸세요. REF는 브랜치·태그·커밋, CONFIG_FILE은 사용할 YAML 경로입니다. 저장소에는 공통 코드가 한 번 포함되어 있어야 합니다.


In [ ]:
REPOSITORY = "c-peace/ensemble_banchmark_pipeline"
REF = "main"
CONFIG_FILE = "configs/experiments/smoke.yaml"

import os
import re
import subprocess
import sys
import tempfile
from pathlib import Path


def secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
    except ImportError:
        return None
    try:
        return userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        return None


if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", REPOSITORY):
    raise ValueError("REPOSITORY는 GitHub의 owner/repository 형식입니다.")
if not REF or REF.startswith("-") or not re.fullmatch(r"[A-Za-z0-9_./-]+", REF):
    raise ValueError("REF에 브랜치, 태그 또는 커밋을 입력하세요.")
CODE_ROOT = Path("/content/ensemble-code")
url = f"https://github.com/{REPOSITORY}.git"
with tempfile.TemporaryDirectory() as temporary:
    askpass = Path(temporary) / "askpass.py"
    askpass.write_text("#!/usr/bin/env python3\nimport os, sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ.get('GITHUB_TOKEN', ''))\n")
    askpass.chmod(0o700)
    git_env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_ASKPASS=str(askpass),
                   GITHUB_TOKEN=secret("GITHUB_TOKEN") or "")

    def git(*args):
        result = subprocess.run(["git", *args], env=git_env, capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError("GitHub 다운로드 실패: 저장소·REF·토큰 권한을 확인하세요.")
        return result.stdout.strip()

    if not CODE_ROOT.exists():
        # 실패한 다운로드가 다음 실행에 남지 않도록 임시 폴더에서 준비합니다.
        checkout = Path(temporary) / "checkout"
        git("init", "--quiet", str(checkout))
        git("-C", str(checkout), "remote", "add", "origin", url)
        git("-C", str(checkout), "fetch", "--quiet", "--depth", "1", "origin", REF)
        git("-C", str(checkout), "checkout", "--quiet", "--detach", "FETCH_HEAD")
        import shutil
        shutil.move(str(checkout), str(CODE_ROOT))
    else:
        if git("-C", str(CODE_ROOT), "remote", "get-url", "origin") != url:
            raise ValueError("다른 저장소가 이미 있습니다. 새 Colab 런타임에서 실행하세요.")
        if git("-C", str(CODE_ROOT), "status", "--porcelain"):
            raise ValueError("가져온 코드가 수정되었습니다. 변경을 저장하고 새 런타임에서 실행하세요.")
        git("-C", str(CODE_ROOT), "fetch", "--quiet", "--depth", "1", "origin", REF)
        if git("-C", str(CODE_ROOT), "rev-parse", "HEAD") != git("-C", str(CODE_ROOT), "rev-parse", "FETCH_HEAD"):
            raise ValueError("GitHub 코드가 변경되었습니다. 새 런타임에서 다시 시작하세요.")
    CODE_COMMIT = git("-C", str(CODE_ROOT), "rev-parse", "HEAD")
os.chdir(CODE_ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", ".[colab]"], check=True)
print("코드 준비 완료:", CODE_COMMIT)

## 2번 셀 · GCP에서 train과 validation 받기
YAML의 data.gcs_uri를 gs://내버킷/prepared로 바꾸세요. 해당 폴더 안에 train/과 validation/이 있어야 합니다. 이미 받은 파일은 체크섬이 일치하면 재사용합니다.


In [ ]:
from google.colab import auth
from ensemble_benchmark.cloud import download_data
from ensemble_benchmark.config import load_config

config = load_config(CODE_ROOT / CONFIG_FILE)
auth.authenticate_user(project_id=config["data"]["gcp_project"])
DATA_ROOT = download_data(
    config["data"]["gcs_uri"], "/content/ensemble-data",
    project=config["data"]["gcp_project"],
)
TRAIN_ROOT = DATA_ROOT / "train"
VALID_ROOT = DATA_ROOT / "validation"
print("Train:", TRAIN_ROOT)
print("Validation:", VALID_ROOT)

## 3번 셀 · YAML 설정을 W&B에 기록하기
YAML의 experiment에서 실험 이름과 W&B 프로젝트를 정합니다. 이 셀을 다시 실행하면 새 실험이 시작됩니다. 6번 셀로 종료하기 전에는 4번 셀만 다시 실행해 학습을 이어갈 수 있습니다. 6번 셀로 종료한 뒤에는 3번 셀에서 새 실험을 시작하세요.


In [ ]:
import uuid
import wandb
import yaml

CHECKPOINT = None
REPORT_PATH = None
experiment = config["experiment"]
if experiment["wandb_mode"] == "online":
    wandb.login(key=secret("WANDB_API_KEY"))
if wandb.run is not None:
    wandb.run.finish()
RUN_ID = f'{experiment["name"]}-{uuid.uuid4().hex[:8]}'
RUN_DIR = Path("/content/ensemble-runs") / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
CONFIG_SNAPSHOT = RUN_DIR / "config.yaml"
CONFIG_SNAPSHOT.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))
run = wandb.init(
    project=experiment["wandb_project"], entity=experiment["wandb_entity"],
    name=RUN_ID, id=RUN_ID, mode=experiment["wandb_mode"], dir=str(RUN_DIR),
    config={**config, "github_repository": REPOSITORY, "github_commit": CODE_COMMIT},
)
print("실험:", RUN_ID)
print("결과 폴더:", RUN_DIR)

## 4번 셀 · 학습하기
기존 모델로 실행하는 예제입니다. 자신의 학습 코드로 바꿔도 됩니다. 학습 루프에서 run.log({"train/loss": float(loss)})를 호출하면 W&B에 그래프가 남습니다. 기본 예제는 아래 콜백으로 기록합니다.


In [ ]:
import torch
from ensemble_benchmark import runtime

if config != yaml.safe_load(CONFIG_SNAPSHOT.read_text()):
    raise ValueError("설정이 변경되었습니다. 3번 셀에서 새 실험을 시작하세요.")
CHECKPOINT = None
REPORT_PATH = None
if "evaluation" in run.summary.keys():
    del run.summary["evaluation"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
release = runtime.build_local_release(
    DATA_ROOT, smoke=config["data"]["smoke"],
    excluded_train_ids=config["data"]["excluded_train_ids"],
)


def log_training(event):
    if event["type"] == "train":
        run.log({"global_step": event["global_step"], "epoch": event["epoch"] + 1,
                 **{f"train/{name}": value for name, value in event["losses"].items()}})


CHECKPOINT = runtime.train_local(
    release, DATA_ROOT, RUN_DIR, device=DEVICE,
    event_callback=log_training, **config["training"],
)
print("체크포인트:", CHECKPOINT)

## 5번 셀 · 평가하기
YAML의 evaluation.enabled가 true이면 validation으로 평가합니다. smoke 결과는 연결 확인용이며 전체 모델 성능 점수가 아닙니다.


In [ ]:
if config != yaml.safe_load(CONFIG_SNAPSHOT.read_text()):
    raise ValueError("설정이 변경되었습니다. 3번 셀에서 새 실험을 시작하세요.")
REPORT_PATH = None
if "evaluation" in run.summary.keys():
    del run.summary["evaluation"]
if config["evaluation"]["enabled"]:
    evaluation = dict(config["evaluation"])
    evaluation.pop("enabled")
    report, REPORT_PATH = runtime.evaluate_local(
        release, DATA_ROOT, RUN_DIR, device=DEVICE, **evaluation,
    )
    run.summary["evaluation"] = report
    print("평가 결과:", REPORT_PATH)
else:
    print("평가 생략: evaluation.enabled=false")

## 6번 셀 · W&B에 모델과 결과 저장하기
학습·평가가 끝나면 실행하세요. 모델, 실제 사용한 YAML, 평가 JSON을 하나의 Artifact로 보관합니다. 자신의 학습 코드에서는 CHECKPOINT와 REPORT_PATH를 저장한 파일 경로로 맞추면 됩니다. Colab 종료 전에 이 셀을 실행해야 모델이 업로드됩니다.


In [ ]:
import hashlib
import json

if config != yaml.safe_load(CONFIG_SNAPSHOT.read_text()):
    raise ValueError("설정이 변경되었습니다. 3번 셀에서 새 실험을 시작하세요.")
if CHECKPOINT is None or not Path(CHECKPOINT).is_file():
    raise ValueError("먼저 학습한 모델 파일을 CHECKPOINT에 지정하세요.")
if REPORT_PATH is not None:
    saved_report = json.loads(Path(REPORT_PATH).read_text())
    if "checkpoint_sha256" in saved_report:
        with Path(CHECKPOINT).open("rb") as stream:
            current_hash = hashlib.file_digest(stream, "sha256").hexdigest()
        if saved_report["checkpoint_sha256"] != current_hash:
            raise ValueError("평가한 모델과 현재 모델이 다릅니다. 평가를 다시 실행하세요.")
artifact = wandb.Artifact(RUN_ID, type="model", metadata={"github_commit": CODE_COMMIT})
artifact.add_file(str(CHECKPOINT), name="checkpoint.pt")
artifact.add_file(str(CONFIG_SNAPSHOT), name="config.yaml")
if REPORT_PATH is not None:
    artifact.add_file(str(REPORT_PATH), name="evaluation.json")
logged = run.log_artifact(artifact)
if experiment["wandb_mode"] == "online":
    logged.wait()
run.finish()
if experiment["wandb_mode"] == "online":
    print("W&B 저장 완료:", run.url)
else:
    print("온라인 업로드 안 함:", experiment["wandb_mode"], "— 로컬 결과:", RUN_DIR)